# 05 Incident reconstruction

## Purpose
Rebuild the INC-001 timeline interactively from the normalized records, so the
published timeline can be checked against the data rather than taken on trust.

## Inputs
- `data/processed/DS002.jsonl`, digest-verified.
- `incidents/INC-001.md`, the published case report.

## Methodology
Only Sysmon `UtcTime` is used to order events. Security and PowerShell records have
no resolved event time and are attached to the timeline by attribute correlation,
never by clock. Every step is labelled with whether it is ordered or attributed, and
the labelled steps are what the report publishes.

## Outputs
A printed, ordered timeline with an ordered-or-attributed marker per row, and the
corroboration count per process-creation step. The durable artefact is
`incidents/INC-001.md`.

## Conclusion
Written at the end of the notebook.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name != "Cyberion-ThreatShield":
    ROOT = Path(r"{root}")
VENV = (ROOT / ".venv/Scripts/python.exe").resolve()
if Path(sys.executable).resolve() != VENV:
    raise RuntimeError(
        "Select the Cyberion (.venv) kernel. This analysis must run on the pinned "
        f"environment, not {sys.executable}."
    )
sys.path.insert(0, str(ROOT / "scripts"))

import test_all as harness

In [ ]:
frames = harness.load_datasets(ROOT)
ds2 = frames["DS002"]
ordered = ds2[ds2["timestamp_utc"].notna()]

lines = [1057, 1073, 2901, 2513, 1364, 1366, 1226, 1238, 2955, 2877, 2367, 1209, 1213]
print("INC-001 evidence lines")
for number in lines:
    matches = ds2[ds2["record_locator"] == number]
    if matches.empty:
        continue
    row = matches.iloc[0]
    record = row["raw"]
    stamp = row["timestamp_utc"] or "UNRESOLVED"
    detail = (record.get("Image") or record.get("NewProcessName")
              or record.get("ServiceName") or record.get("TargetUserName")
              or record.get("DestAddress") or record.get("DestPort") or "")
    print(f"  {stamp:<30} DS002:{number:<5} {row['provider'].split('Windows-')[-1]:<22} "
          f"eid={row['event_id']:<5} {str(detail)[:60]}")

In [ ]:
# The chain that carries the case: is the recorded parent of each step the GUID of
# the process that ran before it? Join on ProcessGuid only, inside one dataset,
# on one exact host string.
def guid_of(line):
    return ds2[ds2["record_locator"] == line].iloc[0]["raw"].get("ProcessGuid")

pairs = [(1366, 1057), (1057, 1073), (1073, 2901)]
for parent_line, child_line in pairs:
    parent = ds2[ds2["record_locator"] == parent_line].iloc[0]
    child = ds2[ds2["record_locator"] == child_line].iloc[0]
    parent_guid = guid_of(parent_line)
    child_parent_guid = child["raw"].get("ParentProcessGuid")
    match = parent_guid == child_parent_guid
    delay = ""
    if parent["timestamp_utc"] and child["timestamp_utc"]:
        delay = f"  delay {child['timestamp_utc']} minus parent"
    print(f"  DS002:{parent_line} -> DS002:{child_line}: "
          f"GUID join {'HOLDS' if match else 'FAILS'}{delay}")

In [ ]:
# Corroboration: does an independent log source agree on the parent relationships?
security = ds2[ds2["provider"] == "Microsoft-Windows-Security-Auditing"]
creations = security[security["event_id"] == 4688]
print("Security 4688 independent process creations")
for _, row in creations.iterrows():
    record = row["raw"]
    print(f"  DS002:{row['record_locator']:<5} {record.get('NewProcessName')} "
          f"parent={record.get('ParentProcessName')}")

## Conclusion

The timeline reconstructs cleanly, and the three GUID joins that carry the case
all hold. The service execution is not a single-log inference: the same parent
relationships appear in Sysmon event 1 and in Security 4688, and the 5156 firewall
record independently attributes the outbound flow to a PowerShell application.

The reconstruction also shows what cannot be concluded. The service installation
at DS002:1213 and the network logon at DS002:1209 both carry no resolved event
time, so the persistence mechanism cannot be placed on this timeline by clock. The
account that created it is known; whether that account was stolen is not, because
the recording contains no authentication-failure records at all. Both facts are
reported as unresolved in the case report rather than being smoothed over.